# Lab 1 Practical ML foundations

We will build a small handwritten-digit classifier, compare it with simple baselines, and learn how to train and evaluate it on a GPU.

Choose **Runtime → Change runtime type → T4 GPU** (or another available GPU) before starting. Changing runtime can reset variables and temporary files. We will configure device placement when we reach training. CPU can be used as a fallback in this lab.

Lab 1 has **no assignments, report, homework or submissions**. It is supposed to be an introductory to popular ML libraries and common workflow that will be used in later tasks.

## Building and evaluating a classifier
### A1. Basic libraries
Our task is to predict a digit from a small grayscale image and learn how to use ML libraries in this process. NumPy stores numeric arrays, pandas makes data summaries readable, Matplotlib visualizes images and learning curves, scikit-learn supplies classical ML tools, and PyTorch supplies differentiable neural networks.

**NumPy: inspect shape and dtype.** A batch has a sample dimension and a feature dimension. Here are two examples and three numeric features describing their shape, data type and means.

In [ ]:
import numpy as np

example_array = np.array(
    [[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]],
    dtype=np.float32,
)

print("Shape:", example_array.shape)
print("Dtype:", example_array.dtype)
print("Feature means:", example_array.mean(axis=0)) # change axis = 1 to see what happens

**pandas: put the same values in a data frame.** It makes it easier to inspect and operate data.

In [ ]:
import pandas as pd

example_table = pd.DataFrame(
    example_array,
    columns=["feature_0", "feature_1", "feature_2"],
)
example_table

**Matplotlib: visualization.** We will use plots for images, class counts, losses and errors.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4, 2))
ax.plot([1, 2, 3], [1, 4, 2], marker="o")
ax.set(xlabel="X", ylabel="Y")
fig.tight_layout()
plt.show()

**PyTorch: tensors and automatic differentiation.** A tensor has a shape, dtype and device. It can also track gradients. For now we use CPU tensors, and later we will explicitly move neural computation to CUDA.

In [ ]:
import torch
from torch import nn

example_tensor = torch.from_numpy(example_array)

print("Shape:", example_tensor.shape)
print("Dtype:", example_tensor.dtype)
print("Device:", example_tensor.device)

**A tiny gradient example.** For a scalar $f(w) = \sum_i w_i x_i$, the gradient with respect to $w$ is $x$. Predict the result, then inspect it. Gradients accumulate unless cleared.

In [ ]:
features = torch.tensor([2.0, -3.0])
weights = torch.tensor([1.0, 2.0], requires_grad=True)

value = (weights * features).sum()
value.backward()

print("Scalar value:", value.item())
print("Gradient:", weights.grad)
assert torch.allclose(weights.grad, features) # validate that these are close

### A2. Data setup
**Digits dataset from scikit-learn.** Each observation is an 8 × 8 grayscale image with a label from 0 to 9. The flat array contains 64 pixel features per example.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()

pixel_features = digits.data
image_labels = digits.target

print(digits.DESCR)

**Inspect the dataset.** Check dimensions, value ranges, label classes and finite values. It is important to understand your data before applying any complicated models. You oftentimes catch some downstream training errors from this stage.

In [ ]:
print("Images:", digits.images.shape)
print("Flat features:", pixel_features.shape)
print("Labels:", image_labels.shape)
print("Pixel range:", pixel_features.min(), pixel_features.max())
print("Classes:", np.unique(image_labels))

assert np.isfinite(pixel_features).all()
assert pixel_features.shape[1] == 64
assert set(np.unique(image_labels)) == set(range(10))

**Analyze class balance.** A high accuracy can be misleading when one class dominates. The class-count table will also tell us what a majority-class baseline might achieve.

In [ ]:
class_counts = (
    pd.Series(image_labels, name="digit").value_counts().sort_index().rename("count")
)

display(class_counts.to_frame())

**Look at representative images.** The same sample can be represented as an 8 × 8 image for plotting or a length-64 feature vector for a fully connected model. Reshaping must preserve sample and pixel order.

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(10, 2))

for image_id, ax in enumerate(axes):
    ax.imshow(digits.images[image_id], cmap="gray", vmin=0, vmax=16)
    ax.set_title(f"ID {image_id}: {image_labels[image_id]}")
    ax.axis("off")

fig.tight_layout()
plt.show()

**Process the pixel features.** Divide by the maximum pixel value of 16 and produce raw features in $[0, 1]$. This fixed conversion does not learn statistics from validation or test examples. The StandardScaler in the next steps does learn statistics, so it must be fitted on training data only.

In [ ]:
raw_features = (pixel_features / 16.0).astype(np.float32)
labels = image_labels.astype(np.int64)
sample_ids = np.arange(len(labels))

assert 0.0 <= raw_features.min() <= raw_features.max() <= 1.0
print(raw_features.min(), raw_features.max())

**Train/validation/test split.** We use a fixed split: 1,077 training, 360 validation and 360 test examples. We use training set to update model parameters, validation set to support decisions and monitor performance, and test set to measure against a frozen pipeline.

In [ ]:
from sklearn.model_selection import train_test_split

SPLIT_SEED = 2223 # for data split only
# stratify ensures that the train and test sets preserve approximately the same class distribution as the original dataset
train_ids, holdout_ids = train_test_split(
    sample_ids,
    test_size=720,
    stratify=labels,
    random_state=SPLIT_SEED
)
val_ids, test_ids = train_test_split(
    holdout_ids,
    test_size=360,
    stratify=labels[holdout_ids],
    random_state=SPLIT_SEED,
)

train_ids, val_ids, test_ids = (np.sort(ids) for ids in (train_ids, val_ids, test_ids))

print("Split sizes:", len(train_ids), len(val_ids), len(test_ids))

**Verify class counts.** Original sample IDs let us trace labels back to images. We can inspect the split design to ensure we have a reasonable testbed for the model.

In [ ]:
# np.bincount counts how many times each non-negative integer appears in an array
split_counts = pd.DataFrame(
    {
        "train": np.bincount(labels[train_ids], minlength=10),
        "validation": np.bincount(labels[val_ids], minlength=10),
        "test": np.bincount(labels[test_ids], minlength=10),
    }
)
display(split_counts)

**Scaling the data.** The validation and test arrays use the same learned means and scales as the training data. Constant pixel features are handled by StandardScaler without division by zero. We scale the data as $\displaystyle z = \frac{x-\mu}{\sigma}$ so they have approximately zero mean and unit variance.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_features = scaler.fit_transform(raw_features[train_ids]).astype(np.float32)
val_features = scaler.transform(raw_features[val_ids]).astype(np.float32)
test_features = scaler.transform(raw_features[test_ids]).astype(np.float32)

print("Prepared training features:", train_features.shape)

**Convert arrays to PyTorch datasets.** TensorDataset keeps a feature vector, integer label and original ID together. These tensors stay on CPU until we explicitly move a batch to GPU.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def make_dataset(features, ids):
    return TensorDataset(
        torch.from_numpy(features),
        torch.from_numpy(labels[ids]),
        torch.from_numpy(ids),
    )


train_dataset = make_dataset(train_features, train_ids)
val_dataset = make_dataset(val_features, val_ids)
test_dataset = make_dataset(test_features, test_ids)

In [ ]:
# you can inspect an individual sample
train_dataset[0]

### A3. Define a neural classifier and establish baselines
**Define a model with nn.Module.** The `__init__` constructor defines reusable components and basic architecture. The `forward` function defines the computations. The default example is a small one-hidden-layer MLP.

The model accepts standardized features of shape `[B, 64]` and returns ten raw logits per example. Do not apply softmax in forward when training with CrossEntropyLoss.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=64, hidden_dims=(32,), num_classes=10):
        super().__init__() # remember this
        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend(
                [
                    nn.Linear(previous_dim, hidden_dim), # linear layer
                    nn.ReLU(), # non-linear activation
                ]
            )
            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, features):
        return self.network(features)

**Check the interface before training.** This temporary prototype only demonstrates shapes.

In [ ]:
prototype = MLP(input_dim=64, hidden_dims=(32,), num_classes=10)

with torch.no_grad():
    prototype_logits = prototype(torch.zeros(4, 64))

print("Batch logits:", prototype_logits.shape)
assert prototype_logits.shape == (4, 10)
del prototype

**Construct baselines for comparison.** Before training the neural model, fit simple baselines on the same training dataset. Both baselines run on CPU.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline

**Majority-class baseline.** Always predicting the most common training label tests whether the learned classifier adds value beyond class prevalence.

In [ ]:
majority_baseline = DummyClassifier(strategy="most_frequent")
majority_baseline.fit(raw_features[train_ids], labels[train_ids])

majority_val_predictions = majority_baseline.predict(raw_features[val_ids])

**Logistic regression baseline.** We can build a stronger baseline with scikit-learn.

In [ ]:
linear_baseline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=500, C=1.0),
)
linear_baseline.fit(raw_features[train_ids], labels[train_ids])

linear_val_predictions = linear_baseline.predict(raw_features[val_ids])

**Evaluate the baselines on validation.** We use accuracy and macro F1 as the metrics.

*   Accuracy
$$
\text{accuracy} = \frac{\text{Number of correct predictions}}{\text{Total predictions}}
$$
*   Macro F1
For each class,
$$
\text{Precision} = \frac{TP}{TP + FP}, \quad \text{Recall} = \frac{TP}{TP + FN}
$$
and F1
$$
F1 = 2\frac{\text{Precision}\cdot \text{Recall}}{\text{Precision} + \text{Recall}}.
$$
Macro F1 is the unweighted average of F1 across all classes:
$$
\text{Macro F1} = \frac{1}{K}\sum_{k=1}^K F1_k,
$$
where $K$ is the number of classes.

In [ ]:
def classification_metrics(targets, predictions):
    return {
        "accuracy": accuracy_score(targets, predictions),
        "macro_f1": f1_score(
            targets,
            predictions,
            labels=list(range(10)),
            average="macro",
            zero_division=0,
        ),
    }


baseline_validation = {
    "Majority": classification_metrics(labels[val_ids], majority_val_predictions),
    "Logistic regression": classification_metrics(labels[val_ids], linear_val_predictions),
}
display(pd.DataFrame(baseline_validation).T)

### A4. Prepare for reproducible training
**Choose and inspect the device.** If CUDA is unavailable, the example falls back to CPU.

In [ ]:
GPU_AVAILABLE = torch.cuda.is_available()
device = torch.device("cuda" if GPU_AVAILABLE else "cpu")

print("Selected device:", device)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)

if GPU_AVAILABLE:
    print("GPU:", torch.cuda.get_device_name(0))
    torch.cuda.reset_peak_memory_stats(device) # useful for measuring the maximum GPU memory consumed during a particular operation, such as training

**Tensor movement is explicit.** Model parameters, input features and class labels must share a device during the forward/loss computation. DataLoader does not move them automatically.

In [ ]:
cpu_example = torch.tensor([[1.0, 2.0]])
device_example = cpu_example.to(device)

print("Original tensor:", cpu_example.device)
print("Transferred tensor:", device_example.device)
print("For NumPy:", device_example.detach().cpu().numpy())

**Device mismatch.** Tensors should be on the same device for computations.

In [ ]:
if GPU_AVAILABLE:
    demo_layer = nn.Linear(2, 1) # currently on cpu
    try:
        demo_layer(device_example)
    except RuntimeError as error:
        print("Expected device mismatch:", str(error).splitlines()[0])
    demo_layer = demo_layer.to(device)
    with torch.inference_mode():
        print("Aligned output device:", demo_layer(device_example).device)
    del demo_layer
else:
    print("Skipped for CPU fallback.")

**Config training hyperparameters.** Save running hyperparameters and other information to make your experiments trackable and reproducible.

In [ ]:
from dataclasses import asdict, dataclass

@dataclass
class TrainConfig:
    hidden_dims: tuple = (32,)
    learning_rate: float = 0.01
    batch_size: int = 64
    epochs: int = 8
    seed: int = 42

config = TrainConfig()
config

**Randomness control for reproduction.** The split was reproducible with SPLIT_SEED. This seed controls model initialization and training randomness.

In [ ]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(config.seed)

**Create dataloaders.** We create reproducible batch with a controlled generator seed here.

In [ ]:
def make_loaders(batch_size, seed):
    generator = torch.Generator().manual_seed(seed)
    return {
        "train": DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True, # batches are shuffled during training
            generator=generator,
            num_workers=0,
        ),
        "train_eval": DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=False,
        ),
        "validation": DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
        ),
    }

loaders = make_loaders(config.batch_size, config.seed)

**Initialize model, optimizer and define loss.** The optimizer must receive the parameters of the model that will actually be trained.

In [ ]:
set_seed(config.seed)

model = MLP(input_dim=64, hidden_dims=config.hidden_dims, num_classes=10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
loss_function = nn.CrossEntropyLoss()

print("Model parameter device:", next(model.parameters()).device)

**Inspect one forward pass and its loss.** We use a non-shuffled batch so it does not consume the training loader's shuffle sequence.

In [ ]:
batch_features, batch_labels, batch_ids = next(iter(loaders["train_eval"]))
batch_features = batch_features.to(device)
batch_labels = batch_labels.to(device)

with torch.no_grad():
    batch_logits = model(batch_features)
    initial_loss = loss_function(batch_logits, batch_labels)

print("Inputs / logits:", batch_features.shape, batch_logits.shape)
print("Target dtype:", batch_labels.dtype, "initial loss:", initial_loss.item())
assert batch_logits.shape == (len(batch_labels), 10)
assert torch.isfinite(batch_logits).all()

### A5. Train, validate and inspect learning
**Commonly used operations.** Clear old gradients, compute logits (model outputs), compute loss, backpropagate, then update parameters.

In [ ]:
def train_batch(model, batch, optimizer, loss_function, device):
    features, labels, _ = batch
    features = features.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True) # clear old gradients
    logits = model(features) # model pass
    loss = loss_function(logits, labels) # compute loss
    loss.backward() # backpropagate
    optimizer.step() # update parameters

    count = labels.numel()
    correct = (logits.argmax(dim=1) == labels).sum().item()
    return loss.item() * count, correct, count # return useful statistics so we can monitor the training

**One epoch visits all training batches.** Training mode is explicitly set.

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_function, device):
    model.train() # set to train mode
    total_loss, total_correct, total_count = 0.0, 0, 0

    for batch in loader:
        loss_sum, correct, count = train_batch(
            model, batch, optimizer, loss_function, device
        ) # defined above
        total_loss += loss_sum
        total_correct += correct
        total_count += count

    return {
        "cross_entropy": total_loss / total_count,
        "accuracy": total_correct / total_count,
    }

**An evaluate helper function.** It uses evaluation mode and disables autograd, but keeps ordered sample IDs for later error analysis.

In [ ]:
@torch.inference_mode() # no gradients
def evaluate(model, loader, device):
    model.eval() # set to evaluation mode
    loss_sum = 0.0
    all_ids, all_targets, all_probabilities = [], [], []

    for features, targets, ids in loader:
        features, targets = features.to(device), targets.to(device)
        logits = model(features)
        loss_sum += nn.functional.cross_entropy(logits, targets, reduction="sum").item()
        all_ids.extend(ids.tolist())
        all_targets.extend(targets.cpu().tolist())
        all_probabilities.extend(logits.softmax(dim=1).cpu().tolist())

    probabilities = np.asarray(all_probabilities)
    predictions = probabilities.argmax(axis=1)
    return {
        "cross_entropy": loss_sum / len(all_targets),
        **classification_metrics(all_targets, predictions),
        "ids": np.asarray(all_ids),
        "targets": np.asarray(all_targets),
        "predictions": predictions,
        "probabilities": probabilities,
    }

**Checkpoint decision.** We will select the checkpoint with the lowest validation cross entropy.

In [ ]:
import time

history = []
best_validation_loss = float("inf")
best_epoch = 0
best_state = None

**Put the pieces together for eight epochs.** Now we construct the loop: train, validate, log and select a checkpoint.

In [ ]:
training_started = time.perf_counter() # timer

for epoch in range(1, config.epochs + 1):
    train_metrics = train_one_epoch(
        model, loaders["train"], optimizer, loss_function, device
    )
    val_metrics = evaluate(model, loaders["validation"], device)
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_metrics["cross_entropy"],
            "val_loss": val_metrics["cross_entropy"],
            "train_accuracy": train_metrics["accuracy"],
            "val_accuracy": val_metrics["accuracy"],
        }
    )

    if val_metrics["cross_entropy"] < best_validation_loss:
        best_validation_loss = val_metrics["cross_entropy"]
        best_epoch = epoch
        best_state = {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        }

training_seconds = time.perf_counter() - training_started
display(pd.DataFrame(history))

**Restore the selected checkpoint.** This may differ from the final epoch.

In [ ]:
model.load_state_dict(best_state)
model.eval()

mlp_validation = evaluate(model, loaders["validation"], device)
validation_table = pd.DataFrame(
    {
        **baseline_validation,
        "MLP": {name: mlp_validation[name] for name in ("accuracy", "macro_f1")},
    }
).T

print("Best epoch:", best_epoch, "training seconds:", round(training_seconds, 3))
display(validation_table)

**The loss curve.** Cross-entropy can change even when accuracy does not. It's usually helpful to look at optimization progress to diagnose the overfitting/underfitting when something is wrong.

In [ ]:
history_table = pd.DataFrame(history)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(history_table["epoch"], history_table["train_loss"], label="Training")
ax.plot(history_table["epoch"], history_table["val_loss"], label="Validation")
ax.set(xlabel="Epoch", ylabel="Cross-entropy")
ax.legend()
fig.tight_layout()
plt.show()

**Accuracy.** The training curve here was measured during optimization. Without dropout or augmentation it is easier to compare with validation, but it is still not the deterministic score of the selected checkpoint.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(history_table["epoch"], history_table["train_accuracy"], label="Training")
ax.plot(history_table["epoch"], history_table["val_accuracy"], label="Validation")
ax.set(xlabel="Epoch", ylabel="Accuracy")
ax.legend()
fig.tight_layout()
plt.show()

**GPU memory.** Allocated memory tracks live tensors, and reserved memory includes the caching allocator. It's helpful if you want to consider GPU memory management.

In [ ]:
gpu_memory = None

if GPU_AVAILABLE:
    gpu_memory = {
        "allocated_MiB": torch.cuda.memory_allocated(device) / 2**20,
        "reserved_MiB": torch.cuda.memory_reserved(device) / 2**20,
        "peak_allocated_MiB": torch.cuda.max_memory_allocated(device) / 2**20,
    }
    display(pd.DataFrame([gpu_memory]))
else:
    print("CUDA memory measurements are unavailable in the CPU fallback.")

### A6. Evaluation, inference and portable checkpoints
**Define inference for a new raw input.** Training/evaluation loaders already contain standardized features. A new input instead arrives as raw $[0, 1]$ pixels, so this function applies the saved training means/scales exactly once and returns CPU probabilities. Softmax is useful here for interpreting predictions, not inside the training model's forward method.

In [ ]:
@torch.inference_mode()
def predict_raw(model, raw, normalization, device):
    model.eval()
    raw = raw.to(device)
    mean = raw.new_tensor(normalization["mean"])
    scale = raw.new_tensor(normalization["scale"])
    standardized = (raw - mean) / scale # standardize the raw input
    return model(standardized).softmax(dim=1).cpu()

**Try a few training images before opening the test set.** This checks the inference interface safely.

In [ ]:
normalization = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist(),
}
raw_probe = torch.from_numpy(raw_features[train_ids[:4]])
probabilities_probe = predict_raw(model, raw_probe, normalization, device)

assert probabilities_probe.shape == (4, 10)
assert torch.allclose(probabilities_probe.sum(dim=1), torch.ones(4))
print("Predicted digits:", probabilities_probe.argmax(dim=1).tolist())

**Save what is needed to reconstruct the model.** A state_dict contains learned tensors, not the architecture or preprocessing contract. Save the constructor arguments, normalization and configuration as well.

In [ ]:
from pathlib import Path
import uuid

run_directory = Path("lab1_runs") / f"session-{uuid.uuid4().hex[:8]}"
run_directory.mkdir(parents=True, exist_ok=False)
checkpoint_path = run_directory / "classifier.pt"

checkpoint = {
    "state_dict": model.state_dict(),
    "model_kwargs": {
        "input_dim": 64,
        "hidden_dims": config.hidden_dims,
        "num_classes": 10,
    },
    "normalization": normalization,
    "config": asdict(config),
    "best_epoch": best_epoch,
}
torch.save(checkpoint, checkpoint_path)
print("Saved in Colab:", checkpoint_path)

**Load GPU-trained weights on CPU.** `map_location='cpu'` makes the tensor placement explicit. Rebuild the same architecture before loading its state.

In [ ]:
loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

cpu_model = MLP(**loaded["model_kwargs"])
cpu_model.load_state_dict(loaded["state_dict"])
cpu_model.eval()

print("Restored parameter device:", next(cpu_model.parameters()).device)

**Test the frozen model.** The MLP configuration and checkpoint were selected without test scores.

In [ ]:
RUN_FINAL_DEMO = True  # Optional closing demonstration after validation decisions.
final_results = None

**Evaluate the frozen classifiers on the same test IDs.** The test loader is created here. We reuse the same evaluation function rather than writing a second metric implementation.

In [ ]:
if RUN_FINAL_DEMO:
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)
    final_results = evaluate(model, test_loader, device)
    final_rows = {
        "Majority": classification_metrics(
            labels[test_ids], majority_baseline.predict(raw_features[test_ids])
        ),
        "Logistic regression": classification_metrics(
            labels[test_ids], linear_baseline.predict(raw_features[test_ids])
        ),
        "MLP": {key: final_results[key] for key in ("accuracy", "macro_f1")},
    }
    display(pd.DataFrame(final_rows).T)
else:
    print("Final test evaluation is off; continue with the explanations below.")

**Inspect per-class errors.** A confusion matrix adds information that one aggregate score hides. It uses the MLP predictions already obtained above, without retraining or choosing new settings.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

if final_results is not None:
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        final_results["targets"],
        final_results["predictions"],
        labels=list(range(10)),
        colorbar=False,
        ax=ax,
    )
    ax.set_title("Frozen MLP · test split")
    fig.tight_layout()
    plt.show()

**Look at an individual success and failure.** You can inspect the actual output to diagnose more fine-grained failure modes. It is usually helpful when you want to find problems that the evaluation metrics can possibly hide.

In [ ]:
if final_results is not None:
    correct = (final_results["targets"] == final_results["predictions"])
    examples = []
    for name, mask in [("Success", correct), ("Error", ~correct)]:
        positions = np.flatnonzero(mask) # returns indices of non-zero elements in flattened way
        if len(positions):
            examples.append((name, int(positions[0]))) # one example per case
        else:
            print("No", name.lower(), "in this evaluation.")

    fig, axes = plt.subplots(1, len(examples), figsize=(5, 2.5), squeeze=False)
    for ax, (name, position) in zip(axes[0], examples):
        image_id = final_results["ids"][position]
        prediction = final_results["predictions"][position]
        confidence = final_results["probabilities"][position, prediction]
        ax.imshow(raw_features[image_id].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
        ax.set_title(
            f"{name} - ID {image_id}\ntrue={labels[image_id]}, pred={prediction}\np={confidence:.2f}"
        )
        ax.axis("off")
    fig.tight_layout()
    plt.show()

## Recap and questions
We built a classifier from inspected data, established baselines, configured reproducible GPU training, measured validation behavior, and introduced inference, checkpoint and final evaluation.

References: [PyTorch basics](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html), [CUDA semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html), [scikit-learn digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html).